# Lekcija 11 - Agent-prema-Agentu (A2A) protokol


## Postavljanje


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Što je A2A protokol?

**Agent-to-Agent (A2A) protokol** je otvoreni standard koji omogućuje AI agentima da komuniciraju,
otkriju jedni druge i surađuju — čak i kada su izgrađeni na različitim okvirima ili ih hostaju
različite usluge.

Ključni koncepti:

- **Otkriće** – Agent objavljuje *Agent karticu* koja opisuje njegove sposobnosti, što olakšava
  ostalim agentima (ili orkestratorima) da pronađu pravog stručnjaka za zadatak.
- **Slanje poruka** – Agentii razmjenjuju strukturirane poruke putem zajedničkog protokola, tako da
  zahtjev od jednog agenta može biti razumljen i izvršen od strane drugog bez obzira na njegovu
  internu implementaciju.
- **Životni ciklus zadatka** – A2A definira stanja poput *predan*, *radi se*, *završeno*, i
  *neuspjelo*, dajući orkestratoru potpun uvid u napredak delegiranog zadatka.

U ovoj lekciji simuliramo A2A-stil suradnje povezujući tri specijalizirana turistička agenta
u tijek rada gdje svaki agent doprinosi svojim stručnim znanjem i prosljeđuje rezultate sljedećem.


## Stvaranje specijaliziranih turističkih agenata


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Suradnja višestrukih agenata putem tijeka rada

Povezujemo tri agenta u sekvencijalni tijek rada koji oponaša A2A prijenos poruka:

1. **CurrencyExchangeAgent** prima korisnički zahtjev i pruža smjernice o valuti.
2. **ActivityPlannerAgent** prima obogaćeni kontekst i dodaje preporuke za aktivnosti.
3. **TravelManagerAgent** sintetizira oba unosa u konačni putni pregled.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Razumijevanje A2A u produkciji

U produkcijskom okruženju A2A protokol omogućuje moćne scenarije između usluga:

| Mogućnost | Opis |
|---|---|
| **Međuraamvrsna interoperabilnost** | Agent izrađen s jednim frameworkom može delegirati zadatke agentu izrađenom s bilo kojim drugim A2A-kompatibilnim frameworkom, omogućujući stvarnu interoperabilnost između organizacija. |
| **Granice usluga** | Agent može živjeti u odvojenim mikroservisima, cloud regijama ili čak različitim organizacijama, a i dalje besprijekorno surađivati. |
| **Dinamičko otkrivanje** | Orkestrator može u runtime-u upitom registrator Agent Karata pronaći najbolje prikladnog stručnjaka za dani podzadatak. |
| **Streaming i push obavijesti** | A2A podržava Server-Sent Events (SSE) za real-time ažuriranja napretka i push obavijesti za dugotrajne zadatke. |

Radni tijek koji smo gore izgradili pojednostavljena je, u-procesna verzija ovog obrasca. U pravoj
implementaciji svaki bi agent izlagao HTTP endpoint, objavljivao Agent Kartu i komunicirao
putem A2A JSON-RPC protokola.


## Sažetak

U ovoj lekciji naučili ste:

1. **Što je A2A protokol** — otvoreni standard za otkrivanje, razmjenu poruka 
   i upravljanje zadacima među agentima.
2. **Kako kreirati specijalizirane agente** — agent za mjenjačnicu valuta, agent za planiranje aktivnosti
   i orkestratora za upravljanje putovanjima.
3. **Kako povezati agente u radni tijek** — korištenjem `WorkflowBuilder` za modeliranje sekvencijalne
   razmjene poruka među agentima.
4. **Kako A2A radi u produkciji** — omogućavanje suradnje između različitih okvira i servisa
   s dinamičkim otkrivanjem i streaming ažuriranjima.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
